In [5]:
# pip install yfinance pandas numpy matplotlib ta


In [6]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:.2f}".format)


In [7]:
TICKERS = [
    "SPY", "QQQ", "AAPL", "MSFT", "NVDA", "GOOGL", "AMZN"
]

START_DATE = "2018-01-01"


In [17]:
def load_weekly_data(ticker):
    df = yf.Ticker(ticker).history(
        start=START_DATE,
        interval="1d",
        auto_adjust=True
    )

    # Keep only Close and force Series
    close = df["Close"].astype(float)

    # Weekly resample
    close = close.resample("W-FRI").last().dropna()

    return pd.DataFrame({"Close": close})


In [18]:
def compute_weekly_signals(df):
    df = df.copy()

    close = pd.Series(df["Close"].values, index=df.index)

    # Defensive checks
    assert close.ndim == 1, f"Close is not 1D: shape={close.shape}"

    df["ma40"] = close.rolling(40).mean()
    df["rsi"] = ta.momentum.RSIIndicator(close, window=14).rsi()

    df["signal"] = "HOLD"
    df.loc[(close > df["ma40"]) & (df["rsi"] < 70), "signal"] = "BUY"
    df.loc[close < df["ma40"], "signal"] = "SELL"

    return df


In [19]:
def backtest(df):
    df = df.copy()

    df["position"] = 0
    df.loc[df["signal"] == "BUY", "position"] = 1
    df.loc[df["signal"] == "SELL", "position"] = 0
    df["position"] = df["position"].ffill().fillna(0)

    df["weekly_return"] = df["Close"].pct_change()
    df["strategy_return"] = df["position"].shift(1) * df["weekly_return"]

    df["cum_market"] = (1 + df["weekly_return"]).cumprod()
    df["cum_strategy"] = (1 + df["strategy_return"]).cumprod()

    return df


In [20]:
results = {}
summaries = []

for ticker in TICKERS:
    df = load_weekly_data(ticker)
    df = compute_weekly_signals(df)
    df = backtest(df)

    results[ticker] = df

    total_return = df["cum_strategy"].iloc[-1] - 1
    market_return = df["cum_market"].iloc[-1] - 1

    summaries.append({
        "Ticker": ticker,
        "Strategy Return %": total_return * 100,
        "Market Return %": market_return * 100,
        "Outperformed": total_return > market_return
    })

summary_df = pd.DataFrame(summaries)
summary_df


,Ticker,Strategy Return %,Market Return %,Outperformed
0,SPY,118.91,186.62,False
1,QQQ,268.22,304.29,False
2,AAPL,172.12,513.64,False
3,MSFT,192.97,460.91,False
4,NVDA,931.96,3397.05,False
5,GOOGL,216.25,493.10,False
6,AMZN,106.77,280.28,False


In [21]:
weekly_signals = []

for ticker, df in results.items():
    latest = df.iloc[-1]
    weekly_signals.append({
        "Ticker": ticker,
        "Signal": latest["signal"],
        "Price": latest["Close"],
        "RSI": latest["rsi"]
    })

signals_df = pd.DataFrame(weekly_signals)
signals_df.sort_values("Signal")


,Ticker,Signal,Price,RSI
0,SPY,BUY,691.66,64.89
1,QQQ,BUY,621.26,59.78
2,AAPL,BUY,251.49,48.37
4,NVDA,BUY,186.23,56.60
6,AMZN,BUY,233.71,53.10
5,GOOGL,HOLD,326.79,76.70
3,MSFT,SELL,454.95,39.60
